In [0]:
import dlt
from pyspark.sql.functions import col

In [0]:
@dlt.table(
    name="bronze.bronze_curves_zero",
    comment="Ingesta raw del archivo CSV CurvesZero desde el Volume",
    table_properties={
        "delta.columnMapping.mode": "name",
        "delta.minReaderVersion": "2",
        "delta.minWriterVersion": "5"
    }
)
def bronze_curves_zero():
  return (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("sep", ";")     
    .load("/Volumes/dev/bronze/input/curves_zero/")
  )

In [0]:
@dlt.table(
    name="silver.silver_curves_zero",
    comment="Datos de CurvesZero con tipos de dato correctos"
)
def silver_curves_zero ():
  return (
    dlt.read_stream("bronze.bronze_curves_zero")
    .withColumnRenamed("Curve Name", "curve_name")
    .withColumnRenamed("Curve Id", "curve_id")
    .withColumnRenamed("Curve Type", "curve_type")
    .withColumnRenamed("DF Mid", "df_mid")
    .withColumnRenamed("Curve Datetime", "curve_time")
    .withColumnRenamed("Curve Point Date", "curve_point_date")
    .withColumnRenamed("Zero Mid", "zero_mid")    
    .withColumnRenamed("Offset Days", "offset_days")
    .withColumnRenamed("Index Tenor", "index_tenor")
    .withColumnRenamed("Rate Index", "rate_index")
    .withColumnRenamed("Trade Filter", "trade_filter")
    .withColumn("df_mid", col("df_mid").cast("double"))
    .withColumn("curve_time", col("curve_time").cast("timestamp"))
    .withColumn("curve_point_date", col("curve_point_date").cast("date"))
    .withColumn("zero_mid", col("zero_mid").cast("double"))
    .withColumn("offset_days", col("offset_days").cast("integer"))
  )

In [0]:
@dlt.table(
    name="gold.gold_curves_zero",
    comment="Datos de CurvesZero listo para entregar a blx_delivery"
)
def gold_curves_zero ():
  return (
    dlt.read_stream("silver.silver_curves_zero")
    .withColumn("df_mid", col("df_mid").cast("string"))
    .withColumn("curve_time", col("curve_time").cast("string"))
    .withColumn("curve_point_date", col("curve_point_date").cast("string"))
    .withColumn("zero_mid", col("zero_mid").cast("string"))
    .withColumn("offset_days", col("offset_days").cast("string"))
  )